# 01 — Handmade Features: Clustering & Per-Cluster Model Selection

Embedding: `HandmadeEmbedder` (18 hand-crafted features per window).  
Clustering: **KMeans** and **HDBSCAN** on the embeddings (no UMAP — already low-dim).

In [ ]:
# ── Cell 1: Imports & config ─────────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath("../../src"))
sys.path.insert(0, os.path.abspath("src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from data.dataloader import Dataloader
from data.tickers import TICKERS
from embeddings.Handmade_embedder import HandmadeEmbedder
from clustering.kmeans import KMeansClusterer
from clustering.hdbscan import HDBSCANClusterer
from clustering.cluster_analysis import (
    elbow_scores, plot_elbow, plot_clusters_2d, cluster_stats,
)
from evaluation.metrics import regression_metrics, directional_accuracy
from evaluation.compare import ComparisonTable
from models import (
    LSTMModel, GRUModel, CNNModel, TransformerModel,
    LightGBMModel, ArimaModel,
)

# ── Constants ──
WINDOW_SIZE = 60
STEP = 1
RANDOM_STATE = 42
N_CLUSTERS_RANGE = range(2, 11)
N_CV_SPLITS = 5
MIN_CLUSTER_SIZE_FOR_CV = 30

np.random.seed(RANDOM_STATE)

FEATURE_COLS = [
    "open", "high", "low", "close", "volume",
    "sma5", "sma20", "ema12", "ema26", "close_sma20",
    "macd", "macd_signal", "macd_hist", "rsi14",
    "bb_pct", "bb_bw", "atr14", "obv",
]

print(f"Tickers: {len(TICKERS)}  |  Window: {WINDOW_SIZE}  |  Step: {STEP}")

In [ ]:
# ── Cell 2: Load data, sliding windows (all tickers), concat splits ───────────
loader = Dataloader()

dataset = loader.cut_on_windows(
    TICKERS,
    seq_len=WINDOW_SIZE,
    step_size=STEP,
    feature_cols=FEATURE_COLS,
)
print(f"Loaded {len(dataset)} / {len(TICKERS)} tickers")

X_train, y_train, X_val, y_val, X_test, y_test = loader.concat_splits(dataset)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

In [ ]:
# ── Cell 3: Handmade embeddings ──────────────────────────────────────────────
embedder = HandmadeEmbedder()

train_embeddings = embedder.transform(X_train)
val_embeddings   = embedder.transform(X_val)
test_embeddings  = embedder.transform(X_test)

print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val   embeddings: {val_embeddings.shape}")
print(f"Test  embeddings: {test_embeddings.shape}")

In [ ]:
# ── Cell 4: Elbow analysis — choose k for KMeans ─────────────────────────────
scores = plot_elbow(train_embeddings, k_range=N_CLUSTERS_RANGE,
                    random_state=RANDOM_STATE)

# Auto-pick k with highest silhouette
best_idx = int(np.argmax(scores["silhouette"]))
BEST_K = scores["k"][best_idx]
print(f"Best k by silhouette: {BEST_K}  "
      f"(silhouette={scores['silhouette'][best_idx]:.3f})")

In [ ]:
# ── Cell 5: KMeans clustering ────────────────────────────────────────────────
kmeans = KMeansClusterer(n_clusters=BEST_K, random_state=RANDOM_STATE, scale=True)
kmeans.fit(train_embeddings)

km_train_labels = kmeans.labels_
km_val_labels   = kmeans.predict(val_embeddings)
km_test_labels  = kmeans.predict(test_embeddings)

plot_clusters_2d(train_embeddings, km_train_labels,
                 title=f"KMeans k={BEST_K} — train (PCA 2D)")

km_stats = cluster_stats(km_train_labels, y_train)
print(km_stats)

In [ ]:
# ── Cell 6: HDBSCAN clustering ──────────────────────────────────────────────
hdb = HDBSCANClusterer(min_cluster_size=50, scale=True, prediction_data=True)
hdb.fit(train_embeddings)

hdb_train_labels = hdb.labels_
n_clusters_hdb = hdb.n_clusters_
noise_pct = hdb.noise_ratio_ * 100
print(f"HDBSCAN found {n_clusters_hdb} clusters, noise: {noise_pct:.1f}%")

# Approximate predict for val and test, then reassign noise via KNN
_scaler_hdb = StandardScaler().fit(train_embeddings)

from hdbscan import approximate_predict

non_noise_mask = hdb_train_labels != -1
knn_hdb = None
if non_noise_mask.sum() > 0:
    knn_hdb = KNeighborsClassifier(n_neighbors=5)
    knn_hdb.fit(train_embeddings[non_noise_mask], hdb_train_labels[non_noise_mask])

def _hdb_predict_and_fix(embeddings, label=""):
    emb_scaled = _scaler_hdb.transform(embeddings)
    raw_labels, _ = approximate_predict(hdb._hdbscan, emb_scaled)
    raw_labels = np.asarray(raw_labels, dtype=np.int64)
    labels_fixed = raw_labels.copy()
    noise_mask = raw_labels == -1
    if knn_hdb is not None and noise_mask.sum() > 0:
        labels_fixed[noise_mask] = knn_hdb.predict(embeddings[noise_mask])
        print(f"Reassigned {noise_mask.sum()} noise {label} points via KNN")
    return labels_fixed

hdb_val_labels  = _hdb_predict_and_fix(val_embeddings, "val")
hdb_test_labels = _hdb_predict_and_fix(test_embeddings, "test")

if non_noise_mask.sum() > 0:
    plot_clusters_2d(train_embeddings[non_noise_mask],
                     hdb_train_labels[non_noise_mask],
                     title="HDBSCAN — train (PCA 2D, noise excluded)")

hdb_stats = cluster_stats(hdb_train_labels, y_train)
print(hdb_stats)

In [ ]:
# ── Cell 7: Per-cluster model selection ──────────────────────────────────────

def make_model(name: str, input_size: int):
    """Factory: returns a fresh (untrained) model instance."""
    common_nn = dict(input_size=input_size, epochs=30, batch_size=64,
                     lr=1e-3, patience=5)
    factories = {
        "LSTM":        lambda: LSTMModel(hidden_size=64, num_layers=2, **common_nn),
        "GRU":         lambda: GRUModel(hidden_size=64, num_layers=2, **common_nn),
        "CNN":         lambda: CNNModel(num_filters=64, num_layers=2, **common_nn),
        "Transformer": lambda: TransformerModel(d_model=64, nhead=4, **common_nn),
        "LightGBM":    lambda: LightGBMModel(n_estimators=300,
                                              random_state=RANDOM_STATE),
        "ARIMA":       lambda: ArimaModel(p=2, d=0, q=2),
    }
    return factories[name]()

MODEL_NAMES = ["LSTM", "GRU", "CNN", "Transformer", "LightGBM", "ARIMA"]


def select_best_models(X, y, labels, model_names, input_size,
                       X_val=None, y_val=None, val_labels=None,
                       n_splits=N_CV_SPLITS,
                       min_size=MIN_CLUSTER_SIZE_FOR_CV):
    """
    For each cluster, run TimeSeriesSplit CV over all models.
    Pick best by MAE via regression_metrics(). Retrain on full cluster data.
    """
    cluster_ids = sorted(set(labels[labels >= 0]))
    results = {}
    trained_models = {}
    cv_table = ComparisonTable()          # collects per-fold metrics

    for cid in cluster_ids:
        mask = labels == cid
        Xc, yc = X[mask], y[mask]
        n_samples = len(yc)
        print(f"\n--- Cluster {cid}  (n={n_samples}) ---")

        fold_maes = {name: [] for name in model_names}

        if n_samples < min_size:
            print(f"  WARNING: only {n_samples} samples, skipping CV")
            for name in model_names:
                try:
                    m = make_model(name, input_size)
                    m.fit(Xc, yc)
                    mdf = regression_metrics(yc, m.predict(Xc),
                                             model_name=name, cluster=cid)
                    cv_table.add(mdf)
                    fold_maes[name].append(float(mdf["mae"].iloc[0]))
                except Exception as e:
                    print(f"    {name} failed: {e}")
                    fold_maes[name].append(np.inf)
        else:
            tscv = TimeSeriesSplit(n_splits=n_splits)
            for fold, (tr_idx, va_idx) in enumerate(tscv.split(Xc)):
                Xtr, ytr = Xc[tr_idx], yc[tr_idx]
                Xva, yva = Xc[va_idx], yc[va_idx]
                for name in model_names:
                    try:
                        m = make_model(name, input_size)
                        m.fit(Xtr, ytr, X_val=Xva, y_val=yva)
                        mdf = regression_metrics(yva, m.predict(Xva),
                                                 model_name=name, cluster=cid)
                        cv_table.add(mdf)
                        fold_maes[name].append(float(mdf["mae"].iloc[0]))
                    except Exception as e:
                        print(f"    Fold {fold} {name} failed: {e}")
                        fold_maes[name].append(np.inf)

        # Select best model by average MAE across folds
        best_name = min(fold_maes, key=lambda n: np.mean(fold_maes[n]))
        results[cid] = {
            "best_model": best_name,
            "mean_mae":   float(np.mean(fold_maes[best_name])),
        }
        print(f"  Best: {best_name}  mean_MAE={results[cid]['mean_mae']:.6f}")

        # Retrain best model on all cluster train data (val for early stopping)
        best_m = make_model(best_name, input_size)
        if val_labels is not None and X_val is not None:
            vmask = val_labels == cid
            if vmask.sum() > 0:
                best_m.fit(Xc, yc, X_val=X_val[vmask], y_val=y_val[vmask])
            else:
                best_m.fit(Xc, yc)
        else:
            best_m.fit(Xc, yc)
        trained_models[cid] = best_m

        # Training loss visualisation
        best_m.plot_loss(title=f"Cluster {cid} — {best_name}")

    return results, trained_models, cv_table


input_size = X_train.shape[2]

print("\n" + "=" * 60)
print("KMeans per-cluster model selection")
print("=" * 60)
km_results, km_models, km_cv = select_best_models(
    X_train, y_train, km_train_labels, MODEL_NAMES, input_size,
    X_val=X_val, y_val=y_val, val_labels=km_val_labels)

print("\n" + "=" * 60)
print("HDBSCAN per-cluster model selection")
print("=" * 60)
hdb_results, hdb_models, hdb_cv = select_best_models(
    X_train, y_train, hdb_train_labels, MODEL_NAMES, input_size,
    X_val=X_val, y_val=y_val, val_labels=hdb_val_labels)

# ── CV summary across models × clusters ──
print("\nKMeans CV summary (MAE by cluster):")
print(km_cv.summary("mae"))
km_cv.plot_metric("mae", title="KMeans CV — MAE by cluster")

print("\nHDBSCAN CV summary (MAE by cluster):")
print(hdb_cv.summary("mae"))
hdb_cv.plot_metric("mae", title="HDBSCAN CV — MAE by cluster")

In [ ]:
# ── Cell 8: Test-set evaluation ──────────────────────────────────────────────

def evaluate_on_test(X_test, y_test, test_labels, trained_models, tag=""):
    """Predict per-cluster on test set, return metrics via regression_metrics."""
    preds = np.full(len(y_test), np.nan)
    for cid, model in trained_models.items():
        mask = test_labels == cid
        if mask.sum() == 0:
            continue
        preds[mask] = model.predict(X_test[mask])
    preds = np.nan_to_num(preds, nan=0.0)
    return regression_metrics(y_test, preds, model_name=f"Cluster-{tag}"), preds


final_table = ComparisonTable()

# ── Baselines: every model trained on all data ──
print("Training baseline models (all data)…")
baseline_preds = {}
for name in MODEL_NAMES:
    try:
        m = make_model(name, input_size)
        m.fit(X_train, y_train, X_val=X_val, y_val=y_val)
        pred = m.predict(X_test)
        mdf = regression_metrics(y_test, pred, model_name=f"Baseline-{name}")
        final_table.add(mdf)
        baseline_preds[name] = pred
        print(f"  {name}: MAE={mdf['mae'].iloc[0]:.6f}  "
              f"DirAcc={mdf['dir_accuracy'].iloc[0]:.3f}")
        m.plot_loss(title=f"Baseline — {name}")
    except Exception as e:
        print(f"  {name} failed: {e}")

# ── KMeans cluster-model evaluation ──
km_metrics, km_preds = evaluate_on_test(
    X_test, y_test, km_test_labels, km_models, tag="KMeans")
final_table.add(km_metrics)

# ── HDBSCAN cluster-model evaluation ──
hdb_metrics, hdb_preds = evaluate_on_test(
    X_test, y_test, hdb_test_labels, hdb_models, tag="HDBSCAN")
final_table.add(hdb_metrics)

# ── Final comparison ──
print("\n=== Final Comparison ===")
print(final_table.raw()[["mae", "rmse", "mape", "r2", "dir_accuracy", "ic"]].to_string())
final_table.plot_metric("mae", title="Test MAE — baselines vs cluster models")
final_table.plot_metric("dir_accuracy", title="Test Directional Accuracy")

In [ ]:
# ── Cell 9: Cluster interpretation ───────────────────────────────────────────

def interpret_clusters(X_windows, y, labels, results, title_prefix=""):
    """Visualise mean profile, return distribution, volatility per cluster."""
    cluster_ids = sorted(set(labels[labels >= 0]))
    n = len(cluster_ids)
    if n == 0:
        print("No clusters to interpret.")
        return

    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n), squeeze=False)

    close_col = FEATURE_COLS.index("close")

    for i, cid in enumerate(cluster_ids):
        mask = labels == cid
        windows = X_windows[mask]
        returns = y[mask]
        close_profiles = windows[:, :, close_col]

        # Mean profile ± std
        mean_prof = close_profiles.mean(axis=0)
        std_prof = close_profiles.std(axis=0)
        t = np.arange(len(mean_prof))

        ax0 = axes[i, 0]
        ax0.plot(t, mean_prof, "b-", lw=2)
        ax0.fill_between(t, mean_prof - std_prof, mean_prof + std_prof,
                         alpha=0.2)
        best = results.get(cid, {}).get("best_model", "?")
        ax0.set_title(f"{title_prefix}Cluster {cid} — mean close profile\n"
                      f"n={mask.sum()}, best model: {best}")
        ax0.set_xlabel("Time step")

        # Return distribution
        ax1 = axes[i, 1]
        ax1.hist(returns, bins=40, edgecolor="k", alpha=0.7)
        ax1.axvline(returns.mean(), color="r", ls="--",
                    label=f"mean={returns.mean():.5f}")
        ax1.set_title("Return distribution")
        ax1.legend()

        # Volatility
        ax2 = axes[i, 2]
        vols = close_profiles.std(axis=1)
        ax2.hist(vols, bins=40, edgecolor="k", alpha=0.7, color="orange")
        ax2.set_title("Intra-window volatility (close std)")

    plt.tight_layout()
    plt.show()

    # Text summary
    vol_medians = []
    for cid in cluster_ids:
        mask = labels == cid
        vol_medians.append(X_windows[mask][:, :, close_col].std(axis=1).mean())
    vol_global_median = float(np.median(vol_medians))

    print(f"\n{title_prefix}Cluster interpretation:")
    for idx, cid in enumerate(cluster_ids):
        mask = labels == cid
        ret = y[mask]
        vol = vol_medians[idx]
        best = results.get(cid, {}).get("best_model", "?")
        regime = ("trending up" if ret.mean() > 0.001 else
                  "trending down" if ret.mean() < -0.001 else "sideways")
        vol_label = "high" if vol > vol_global_median else "low"
        print(f"  Cluster {cid}: n={mask.sum()}, {regime}, {vol_label} vol, "
              f"best model: {best}")


print("=== KMeans clusters ===")
interpret_clusters(X_train, y_train, km_train_labels, km_results,
                   title_prefix="KMeans ")

print("\n=== HDBSCAN clusters ===")
interpret_clusters(X_train, y_train, hdb_train_labels, hdb_results,
                   title_prefix="HDBSCAN ")